# Predictive Maintenance with XGBoost, IBM DB2, and Maximo

**Version 5.1 (Production Refactor — patched)**

```
Predictive Maintenance with XGBoost, IBM DB2, and Maximo.

This module builds a panel of asset snapshots from Maximo work orders,
trains an XGBoost model to predict failures within a 30-day horizon,
calibrates probabilities, and writes predictions to DB2.

Author:  Principal ML Engineer
Version: 5.1 (Production Refactor - patched)

CHANGES FROM v5.0
------------------------------------------------------------------------------
1. RESTORED three functions that main() called but the uploaded notebook
   never defined: generate_snapshot_dates(), build_snapshot(),
   walk_forward_folds(). Without them main() would fail with NameError
   before reaching training.
2. FIXED the calibration cutoff. v5.1 split panel by 85% of *snapshot dates*:
       calib_cut = dates_sorted[int(len(dates_sorted) * 0.85)]
   Your failure activity is clustered near the recent end of WORKORDER
   history, so that date-count split pushed every positive label past the
   cutoff -- the training slice XGBoost actually saw had zero positives
   (confirmed: your saved model's scale_pos_weight=617721 = 1833 assets x
   337 dates x n_pos=0). Every tree came back a single-node stump because
   there was nothing to split on. Replaced with select_calibration_cutoff(),
   which cuts by cumulative *positive* count instead, and raises immediately
   if either side ends up without enough positives rather than silently
   training on an all-negative set.
3. walk_forward_folds() carries the same positive-aware guard per fold,
   so an early, currently-failure-free period can't silently produce a
   fold that trains or validates on zero positives either.
4. Fixed find_optimal_threshold()'s ROC branch, which called
   roc_auc_score(..., return_roc=True) -- not a real sklearn parameter,
   would have raised TypeError the first time it was actually used.
Everything else is unchanged from your v5.1 upload.
------------------------------------------------------------------------------
```

In [1]:
# ----------------------------------------------------------------------------
# 1. IMPORTS
# ----------------------------------------------------------------------------
import os
import sys
import warnings
import logging
from datetime import timedelta, datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Union

import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import ibm_db_dbi
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    f1_score, recall_score, precision_score, confusion_matrix,
    precision_recall_curve, roc_curve
)
from sklearn.model_selection import ParameterGrid
from sklearn.calibration import CalibratedClassifierCV

warnings.filterwarnings('ignore')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

In [ ]:
# ----------------------------------------------------------------------------
# 2. CONFIGURATION
# ----------------------------------------------------------------------------

# IBM DB2 client configuration (optional)
SQLLIB = os.getenv("DB2_SQLLIB")

if SQLLIB and os.path.exists(SQLLIB):
    if SQLLIB not in os.environ.get("PATH", ""):
        os.environ["PATH"] = os.path.join(SQLLIB, "bin") + os.pathsep + os.environ["PATH"]
    os.environ["IBM_DB_HOME"] = SQLLIB

# Database connection settings
DB_NAME = os.getenv("DB_NAME", "MAXDB76")
DB_SCHEMA = os.getenv("DB_SCHEMA", "MAXIMO")
DB_USER = os.getenv("DB_USER", "")
DB_PASS = os.getenv("DB_PASS", "")
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = int(os.getenv("DB_PORT", "50000"))

# Asset sites
SITES = ["SITE_1", "SITE_2", "SITE_3", "SITE_4"]

FAILURE_WORKTYPES = ("CM", "EM")
PM_WORKTYPE = "PM"
PREDICTION_HORIZON_DAYS = 30

# Feature engineering windows
SHORT_WINDOW_DAYS = 14
LONG_WINDOW_DAYS = 60
MTBF_LOOKBACK_DAYS = 365
SNAPSHOT_FREQ_DAYS = 3

# Risk thresholds and validation
RISK_HIGH_PCTILE = 95
RISK_MEDIUM_PCTILE = 80
LEAKAGE_CORR_THRESHOLD = 0.85

# Model configuration
MODEL_VERSION = "v5.1"

# Output directory
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# SQL filters
SITES_SQL = ", ".join(f"'{s}'" for s in SITES)
FAILURE_WORKTYPES_SQL = ", ".join(
    f"'{w}'" for w in FAILURE_WORKTYPES
)

logger.info(
    f"Configuration loaded: XGBoost {xgb.__version__}, "
    f"Pandas {pd.__version__}"
)

2026-07-12 15:17:49  INFO  Configuration loaded: XGBoost 3.3.0, Pandas 2.3.3


In [3]:
# ----------------------------------------------------------------------------
# 3. DATABASE UTILITIES
# ----------------------------------------------------------------------------
def get_connection() -> ibm_db_dbi.Connection:
    """Create and return a DB2 connection using ibm_db_dbi."""
    dsn = (
        f'DATABASE={DB_NAME};HOSTNAME={DB_HOST};PORT={DB_PORT};'
        f'PROTOCOL=TCPIP;UID={DB_USER};PWD={DB_PASS};'
    )
    return ibm_db_dbi.connect(dsn, '', '')


def run_sql(connection: ibm_db_dbi.Connection, sql: str) -> pd.DataFrame:
    """Execute SQL and return DataFrame with lowercased column names."""
    df = pd.read_sql(sql, connection)
    df.columns = [c.lower() for c in df.columns]
    return df


def date_literal(d: Union[str, datetime, pd.Timestamp]) -> str:
    """Return a DB2 DATE literal string."""
    if isinstance(d, pd.Timestamp):
        d = d.to_pydatetime()
    elif isinstance(d, str):
        d = datetime.strptime(d, '%Y-%m-%d')
    return f"DATE('{d.strftime('%Y-%m-%d')}')"


def safe_merge(left: pd.DataFrame, right: pd.DataFrame, on: List[str], how: str = 'left') -> pd.DataFrame:
    """Merge with validation for key columns existence."""
    for col in on:
        if col not in left.columns:
            raise KeyError(f"Column '{col}' not found in left DataFrame")
        if col not in right.columns:
            raise KeyError(f"Column '{col}' not found in right DataFrame")
    return left.merge(right, on=on, how=how)

In [4]:
# ----------------------------------------------------------------------------
# 4. SQL QUERY BUILDERS
# ----------------------------------------------------------------------------
def sql_assets() -> str:
    """SELECT active assets from the specified sites."""
    return f"""
SELECT a.ASSETNUM, a.SITEID, a.DESCRIPTION, a.INSTALLDATE, a.CHANGEDATE,
       a.LOCATION, a.ASSETTYPE, COALESCE(a.PRIORITY, 2) AS PRIORITY
FROM {DB_SCHEMA}.ASSET a
WHERE a.SITEID IN ({SITES_SQL})
  AND a.STATUS NOT IN ('DECOMMISSIONED','DISPOSED','INACTIVE')
"""


def sql_label(as_of: Union[str, datetime, pd.Timestamp]) -> str:
    """Forward-looking label: failure within horizon after as_of."""
    start = date_literal(as_of)
    end = date_literal(pd.Timestamp(as_of) + timedelta(days=PREDICTION_HORIZON_DAYS))
    return f"""
SELECT DISTINCT wo.ASSETNUM, wo.SITEID, 1 AS LABEL
FROM {DB_SCHEMA}.WORKORDER wo
WHERE wo.SITEID IN ({SITES_SQL})
  AND wo.WORKTYPE IN ({FAILURE_WORKTYPES_SQL})
  AND wo.STATUS NOT IN ('CAN','WAPPR')
  AND wo.REPORTDATE > {start}
  AND wo.REPORTDATE <= {end}
"""


def sql_rolling_wo(as_of: Union[str, datetime, pd.Timestamp], window_days: int, suffix: str) -> str:
    """Aggregated work order counts and costs over a lookback window."""
    as_of_lit = date_literal(as_of)
    return f"""
SELECT
    wo.ASSETNUM, wo.SITEID,
    COUNT(*)                                                        AS WO_COUNT_{suffix},
    SUM(CASE WHEN wo.WORKTYPE IN ({FAILURE_WORKTYPES_SQL}) THEN 1 ELSE 0 END) AS CM_COUNT_{suffix},
    SUM(CASE WHEN wo.WORKTYPE = '{PM_WORKTYPE}' THEN 1 ELSE 0 END)  AS PM_COUNT_{suffix},
    SUM(COALESCE(wo.ACTLABCOST,0) + COALESCE(wo.ACTMATCOST,0))      AS COST_{suffix}
FROM {DB_SCHEMA}.WORKORDER wo
WHERE wo.SITEID IN ({SITES_SQL})
  AND wo.STATUS NOT IN ('CAN','WAPPR')
  AND wo.REPORTDATE >= {as_of_lit} - {window_days} DAYS
  AND wo.REPORTDATE <  {as_of_lit}
GROUP BY wo.ASSETNUM, wo.SITEID
"""


def sql_mttr(as_of: Union[str, datetime, pd.Timestamp], lookback_days: int = 730) -> str:
    """Average MTTR (hours) for historical failures."""
    as_of_lit = date_literal(as_of)
    return f"""
SELECT wo.ASSETNUM, wo.SITEID,
       AVG(TIMESTAMPDIFF(8, CHAR(wo.ACTFINISH - wo.ACTSTART))) AS MTTR_HOURS
FROM {DB_SCHEMA}.WORKORDER wo
WHERE wo.SITEID IN ({SITES_SQL})
  AND wo.WORKTYPE IN ({FAILURE_WORKTYPES_SQL})
  AND wo.STATUS NOT IN ('CAN','WAPPR')
  AND wo.ACTSTART IS NOT NULL AND wo.ACTFINISH IS NOT NULL
  AND wo.REPORTDATE >= {as_of_lit} - {lookback_days} DAYS
  AND wo.REPORTDATE <  {as_of_lit}
GROUP BY wo.ASSETNUM, wo.SITEID
"""


def sql_failure_history(as_of: Union[str, datetime, pd.Timestamp], lookback_days: int) -> str:
    """Raw failure dates for MTBF calculation."""
    as_of_lit = date_literal(as_of)
    return f"""
SELECT wo.ASSETNUM, wo.SITEID, wo.REPORTDATE AS FAILURE_DATE
FROM {DB_SCHEMA}.WORKORDER wo
WHERE wo.SITEID IN ({SITES_SQL})
  AND wo.WORKTYPE IN ({FAILURE_WORKTYPES_SQL})
  AND wo.STATUS NOT IN ('CAN','WAPPR')
  AND wo.REPORTDATE >= {as_of_lit} - {lookback_days} DAYS
  AND wo.REPORTDATE <  {as_of_lit}
ORDER BY wo.ASSETNUM, wo.SITEID, wo.REPORTDATE
"""


def sql_pm_compliance(as_of: Union[str, datetime, pd.Timestamp], lookback_days: int = 365) -> str:
    """Proportion of PM work orders finished on or before scheduled finish."""
    as_of_lit = date_literal(as_of)
    return f"""
SELECT wo.ASSETNUM, wo.SITEID,
       AVG(CASE WHEN wo.ACTFINISH <= wo.SCHEDFINISH THEN 1.0 ELSE 0.0 END) AS PM_COMPLIANCE_RATIO
FROM {DB_SCHEMA}.WORKORDER wo
WHERE wo.SITEID IN ({SITES_SQL})
  AND wo.WORKTYPE = '{PM_WORKTYPE}'
  AND wo.STATUS NOT IN ('CAN','WAPPR')
  AND wo.ACTFINISH IS NOT NULL AND wo.SCHEDFINISH IS NOT NULL
  AND wo.REPORTDATE >= {as_of_lit} - {lookback_days} DAYS
  AND wo.REPORTDATE <  {as_of_lit}
GROUP BY wo.ASSETNUM, wo.SITEID
"""

In [5]:
# ----------------------------------------------------------------------------
# 5. FEATURE ENGINEERING (Pure pandas, vectorized)
# ----------------------------------------------------------------------------
SHORT_SUFFIX = f'{SHORT_WINDOW_DAYS}d'
LONG_SUFFIX = f'{LONG_WINDOW_DAYS}d'

def compute_mtbf_features(
    failure_df: pd.DataFrame,
    asset_key: pd.DataFrame,
    as_of: pd.Timestamp
) -> pd.DataFrame:
    """
    Compute MTBF, failure count, and days since last failure.
    All operations are vectorized per asset.
    """
    if failure_df.empty:
        result = asset_key.copy()
        result['mtbf_days'] = np.nan
        result['total_failures_hist'] = 0
        result['days_since_last_failure'] = 9999
        result['has_failure_history'] = False
        return result

    failure_df = failure_df.copy()
    failure_df['failure_date'] = pd.to_datetime(failure_df['failure_date'])
    failure_df = failure_df.sort_values(['assetnum', 'siteid', 'failure_date'])

    agg = failure_df.groupby(['assetnum', 'siteid'], as_index=False).agg(
        total_failures_hist=('failure_date', 'count'),
        last_failure_date=('failure_date', 'max')
    )

    failure_df['prev_date'] = failure_df.groupby(['assetnum', 'siteid'])['failure_date'].shift(1)
    failure_df['gap_days'] = (failure_df['failure_date'] - failure_df['prev_date']).dt.days
    mtbf = failure_df.groupby(['assetnum', 'siteid'], as_index=False)['gap_days'].mean()
    mtbf.rename(columns={'gap_days': 'mtbf_days'}, inplace=True)

    result = agg.merge(mtbf, on=['assetnum', 'siteid'], how='left')

    result['days_since_last_failure'] = (as_of - result['last_failure_date']).dt.days
    result['days_since_last_failure'] = result['days_since_last_failure'].fillna(9999).astype(int)

    result['has_failure_history'] = result['total_failures_hist'] > 0
    result.drop(columns=['last_failure_date'], inplace=True)

    result = asset_key.merge(result, on=['assetnum', 'siteid'], how='left')
    result['mtbf_days'] = result['mtbf_days'].fillna(np.nan)
    result['total_failures_hist'] = result['total_failures_hist'].fillna(0).astype(int)
    result['days_since_last_failure'] = result['days_since_last_failure'].fillna(9999).astype(int)
    result['has_failure_history'] = result['has_failure_history'].fillna(False).astype(bool)

    return result


def engineer_snapshot(
    assets_df: pd.DataFrame,
    short_wo: pd.DataFrame,
    long_wo: pd.DataFrame,
    mttr_df: pd.DataFrame,
    failure_df: pd.DataFrame,
    pm_df: pd.DataFrame,
    as_of: pd.Timestamp
) -> pd.DataFrame:
    """Build a feature-rich snapshot for each asset at a given as_of date."""
    key = assets_df[['assetnum', 'siteid']].drop_duplicates()
    if key.empty:
        raise ValueError('No assets found for the given sites.')

    df = key.copy()
    df['as_of_date'] = as_of

    df = safe_merge(df, short_wo, on=['assetnum', 'siteid'], how='left')
    df = safe_merge(df, long_wo, on=['assetnum', 'siteid'], how='left')
    df = safe_merge(df, mttr_df, on=['assetnum', 'siteid'], how='left')
    df = safe_merge(df, pm_df, on=['assetnum', 'siteid'], how='left')

    mtbf_features = compute_mtbf_features(failure_df, key, as_of)
    df = safe_merge(df, mtbf_features, on=['assetnum', 'siteid'], how='left')

    asset_meta = assets_df[['assetnum', 'siteid', 'installdate', 'changedate', 'priority']].copy()
    df = safe_merge(df, asset_meta, on=['assetnum', 'siteid'], how='left')

    df['installdate'] = pd.to_datetime(df['installdate'], errors='coerce')
    df['changedate'] = pd.to_datetime(df['changedate'], errors='coerce')
    age = (as_of - df['installdate']).dt.days
    fallback = (as_of - df['changedate']).dt.days
    df['asset_age_days'] = age.where(age.notna(), fallback).clip(lower=0)
    median_age = df['asset_age_days'].median()
    df['asset_age_days'] = df['asset_age_days'].fillna(median_age if pd.notna(median_age) else 1825)

    df['criticality_score'] = np.where(df['priority'] == 1, 3.0, np.where(df['priority'] == 2, 2.0, 1.0))

    long_count_col = f'wo_count_{LONG_SUFFIX}'
    short_count_col = f'wo_count_{SHORT_SUFFIX}'
    long_count = df[long_count_col]
    short_count = df[short_count_col]
    long_rate = long_count / LONG_WINDOW_DAYS
    short_rate = short_count / SHORT_WINDOW_DAYS

    df['activity_acceleration'] = np.where(
        long_rate > 0, short_rate / long_rate.clip(lower=1e-6), np.nan
    )

    long_cm_col = f'cm_count_{LONG_SUFFIX}'
    df['corrective_ratio_long'] = np.where(
        long_count > 0, df[long_cm_col] / long_count, np.nan
    )

    df['pm_overdue_ratio'] = 1.0 - df['pm_compliance_ratio']
    df['pm_overdue_ratio'] = df['pm_overdue_ratio'].fillna(0)

    df['failure_frequency'] = np.where(
        df['asset_age_days'] > 0,
        df['total_failures_hist'] * 365.0 / df['asset_age_days'],
        0.0
    )

    numeric_cols = [
        f'wo_count_{SHORT_SUFFIX}', f'cm_count_{SHORT_SUFFIX}',
        f'pm_count_{SHORT_SUFFIX}', f'cost_{SHORT_SUFFIX}',
        f'wo_count_{LONG_SUFFIX}', f'cm_count_{LONG_SUFFIX}',
        f'pm_count_{LONG_SUFFIX}', f'cost_{LONG_SUFFIX}',
        'mttr_hours', 'mtbf_days', 'total_failures_hist', 'days_since_last_failure',
        'pm_compliance_ratio', 'asset_age_days', 'criticality_score',
        'activity_acceleration', 'corrective_ratio_long',
        'pm_overdue_ratio', 'failure_frequency'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    return df

In [6]:
# ----------------------------------------------------------------------------
# 6. SNAPSHOT DATES & SINGLE-SNAPSHOT BUILDER  (restored — see header notes)
# ----------------------------------------------------------------------------
def generate_snapshot_dates(
    earliest_wo_date: Union[str, datetime, pd.Timestamp],
    latest_wo_date: Union[str, datetime, pd.Timestamp],
    freq_days: int = SNAPSHOT_FREQ_DAYS,
    horizon_days: int = PREDICTION_HORIZON_DAYS
) -> List[pd.Timestamp]:
    """
    Generate as-of snapshot dates spanning the available WORKORDER history.
    Leaves SHORT_WINDOW_DAYS of runway at the start (so rolling features
    aren't empty by construction) and horizon_days of runway at the end
    (so every snapshot has a fully-realized, non-censored label window).
    """
    start = pd.Timestamp(earliest_wo_date).normalize() + timedelta(days=SHORT_WINDOW_DAYS)
    end = pd.Timestamp(latest_wo_date).normalize() - timedelta(days=horizon_days)
    if end < start:
        raise ValueError(
            f'Not enough WORKORDER history yet: need at least '
            f'{SHORT_WINDOW_DAYS + horizon_days} days between earliest WO and today, '
            f'have {(pd.Timestamp(latest_wo_date) - pd.Timestamp(earliest_wo_date)).days}.'
        )
    return list(pd.date_range(start, end, freq=f'{freq_days}D'))


def build_snapshot(connection: ibm_db_dbi.Connection, as_of: pd.Timestamp) -> pd.DataFrame:
    """
    Pull one full feature+label snapshot for a given as-of date. Features are
    computed strictly BEFORE as_of; the label is computed strictly AFTER it
    (see sql_label / sql_rolling_wo / sql_mttr / sql_failure_history /
    sql_pm_compliance) so the two windows can never overlap.
    """
    assets_df = run_sql(connection, sql_assets())
    label_df = run_sql(connection, sql_label(as_of))
    short_wo = run_sql(connection, sql_rolling_wo(as_of, SHORT_WINDOW_DAYS, SHORT_SUFFIX))
    long_wo = run_sql(connection, sql_rolling_wo(as_of, LONG_WINDOW_DAYS, LONG_SUFFIX))
    mttr_df = run_sql(connection, sql_mttr(as_of))
    failure_df = run_sql(connection, sql_failure_history(as_of, MTBF_LOOKBACK_DAYS))
    failure_df['failure_date'] = pd.to_datetime(failure_df['failure_date'])
    pm_df = run_sql(connection, sql_pm_compliance(as_of))

    snapshot = engineer_snapshot(assets_df, short_wo, long_wo, mttr_df, failure_df, pm_df, as_of)
    snapshot = safe_merge(snapshot, label_df, on=['assetnum', 'siteid'], how='left')
    snapshot['label'] = snapshot['label'].fillna(0).astype(int)
    return snapshot

In [7]:
# ----------------------------------------------------------------------------
# 7. PANEL DATASET BUILDER
# ----------------------------------------------------------------------------
def build_panel(
    connection: ibm_db_dbi.Connection,
    snapshot_dates: List[pd.Timestamp]
) -> pd.DataFrame:
    """Build a panel dataset by iterating over snapshot dates."""
    panels = []
    for as_of in snapshot_dates:
        try:
            snapshot = build_snapshot(connection, as_of)
            panels.append(snapshot)
            pos = snapshot['label'].sum()
            logger.info(f'  {as_of.date()}: {len(snapshot):,} assets, {pos:.0f} positive')
        except Exception as e:
            logger.error(f'Failed to build snapshot for {as_of.date()}: {e}')
            raise
    if not panels:
        raise ValueError('No snapshots built.')
    panel = pd.concat(panels, ignore_index=True)
    logger.info(f'Panel built: {len(panel):,} rows, {panel["assetnum"].nunique():,} unique assets, '
                f'{panel["label"].sum():.0f} positive ({panel["label"].mean()*100:.2f}%)')
    return panel

In [8]:
# ----------------------------------------------------------------------------
# 8. DATA VALIDATION AND LEAKAGE DETECTION
# ----------------------------------------------------------------------------
FEATURE_COLS = [
    f'wo_count_{SHORT_SUFFIX}', f'cm_count_{SHORT_SUFFIX}',
    f'pm_count_{SHORT_SUFFIX}', f'cost_{SHORT_SUFFIX}',
    f'wo_count_{LONG_SUFFIX}',  f'cm_count_{LONG_SUFFIX}',
    f'pm_count_{LONG_SUFFIX}',  f'cost_{LONG_SUFFIX}',
    'mttr_hours', 'mtbf_days', 'total_failures_hist', 'days_since_last_failure',
    'pm_compliance_ratio', 'asset_age_days', 'criticality_score',
    'activity_acceleration', 'corrective_ratio_long',
    'pm_overdue_ratio', 'failure_frequency'
]


def validate_panel(panel: pd.DataFrame) -> None:
    """Run validation checks on the panel before training."""
    logger.info('Validating panel...')
    required_cols = ['assetnum', 'siteid', 'as_of_date', 'label'] + FEATURE_COLS
    missing = [c for c in required_cols if c not in panel.columns]
    if missing:
        raise ValueError(f'Missing columns in panel: {missing}')

    dup = panel.duplicated(subset=['assetnum', 'siteid', 'as_of_date'])
    if dup.any():
        raise ValueError(f'Found {dup.sum()} duplicate rows in panel.')

    if panel['label'].isnull().any():
        raise ValueError('Label contains null values.')

    for col in FEATURE_COLS:
        if panel[col].nunique() == 1:
            logger.warning(f'Feature {col} is constant (value {panel[col].iloc[0]})')

    if panel['label'].sum() == 0:
        raise ValueError('Panel has zero positive labels overall — check sql_label() '
                          'and PREDICTION_HORIZON_DAYS before training anything.')

    logger.info('Validation passed.')


def leakage_guard(panel: pd.DataFrame, threshold: float = LEAKAGE_CORR_THRESHOLD) -> None:
    """Warn if any feature has a correlation with the label exceeding threshold."""
    corr = panel[FEATURE_COLS + ['label']].corr()['label'].drop('label')
    flags = corr[corr.abs() >= threshold]
    if len(flags):
        logger.warning('LEAKAGE GUARD TRIPPED — investigate before training:')
        for feat, c in flags.items():
            logger.warning(f'  {feat}: corr={c:.3f} with label')
    else:
        logger.info(f'Leakage guard clear — max |corr| = {corr.abs().max():.3f}')


def select_calibration_cutoff(
    panel: pd.DataFrame,
    date_col: str = 'as_of_date',
    calib_positive_frac: float = 0.20,
    min_train_positives: int = 30
) -> pd.Timestamp:
    """
    Chronological train/calibration cutoff chosen by cumulative POSITIVE
    count, not by raw date count. This is the fix for the bug that produced
    an all-stump v5.1 model: your failures cluster near the recent end of
    WORKORDER history, so a date-count split (e.g. dates_sorted[int(n*0.85)])
    can push every positive label past the cutoff, leaving zero positives to
    train on. Raises immediately instead of silently proceeding either way.
    """
    dates_sorted = np.sort(panel[date_col].unique())
    pos_by_date = (panel[panel['label'] == 1]
                   .groupby(date_col).size()
                   .reindex(dates_sorted, fill_value=0))
    total_pos = pos_by_date.sum()
    if total_pos == 0:
        raise ValueError('Panel has zero positive labels overall — check sql_label() '
                          'and PREDICTION_HORIZON_DAYS before training anything.')

    cum_frac = pos_by_date.cumsum() / total_pos
    eligible = cum_frac[cum_frac >= (1 - calib_positive_frac)]
    cutoff = eligible.index[0] if len(eligible) else dates_sorted[-2]

    train_pos = int(panel.loc[panel[date_col] <= cutoff, 'label'].sum())
    calib_pos = int(panel.loc[panel[date_col] > cutoff, 'label'].sum())
    if train_pos < min_train_positives:
        raise ValueError(
            f'Only {train_pos} positives would land in training (need >= '
            f'{min_train_positives}). Widen the snapshot date range, shorten '
            f'PREDICTION_HORIZON_DAYS, or lower calib_positive_frac.'
        )
    if calib_pos < 5:
        logger.warning(f'Only {calib_pos} positives in the calibration slice — '
                        f'Platt scaling will be noisy. Consider lowering calib_positive_frac.')
    logger.info(f'Calibration cutoff {pd.Timestamp(cutoff).date()}: '
                f'train_pos={train_pos}, calib_pos={calib_pos}')
    return cutoff

In [9]:
# ----------------------------------------------------------------------------
# 9. CROSS-VALIDATION AND TRAINING UTILITIES
# ----------------------------------------------------------------------------
def walk_forward_folds(
    panel: pd.DataFrame,
    date_col: str = 'as_of_date',
    n_folds: int = 4,
    min_train_frac: float = 0.5,
    min_fold_positives: int = 5
) -> Tuple[List[Tuple[pd.Index, pd.Index]], bool]:
    """
    Expanding-window (walk-forward) folds: each fold trains on every snapshot
    strictly before a cutoff and validates on the next slice after it, so no
    fold can ever see its own future. Folds where either side has fewer than
    min_fold_positives are skipped with a warning rather than silently kept —
    the same class of bug that produced the all-stump v5.1 model can happen
    inside an individual fold, not just at the final calibration split.
    """
    dates = np.sort(panel[date_col].unique())
    n = len(dates)
    start = max(int(n * min_train_frac), 1)
    if n <= start + 1:
        cut = dates[max(int(n * 0.8), 1) - 1]
        train_idx = panel.index[panel[date_col] <= cut]
        val_idx = panel.index[panel[date_col] > cut]
        return [(train_idx, val_idx)], True

    cut_points = sorted(set(np.linspace(start, n - 2, num=min(n_folds, n - start - 1), dtype=int)))
    folds = []
    for i, cp in enumerate(cut_points):
        train_cut = dates[cp]
        next_cut = dates[cut_points[i + 1]] if i + 1 < len(cut_points) else dates[-1]
        tr = panel.index[panel[date_col] <= train_cut]
        va = panel.index[(panel[date_col] > train_cut) & (panel[date_col] <= next_cut)]
        train_pos = panel.loc[tr, 'label'].sum()
        val_pos = panel.loc[va, 'label'].sum()
        if len(va) == 0 or val_pos < min_fold_positives or train_pos < min_fold_positives:
            logger.warning(f'Skipping fold at cutoff {pd.Timestamp(train_cut).date()}: '
                            f'train_pos={train_pos}, val_pos={val_pos}')
            continue
        folds.append((tr, va))
    if not folds:
        raise ValueError('No walk-forward folds have enough positives on both sides. '
                          'Check panel.groupby("as_of_date")["label"].sum().')
    return folds, False


def train_xgboost(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    params: Optional[Dict] = None,
    seed: int = 42
) -> xgb.XGBClassifier:
    """Train an XGBoost classifier with early stopping. Handles class imbalance
    via scale_pos_weight."""
    if params is None:
        params = {
            'n_estimators': 300,
            'max_depth': 4,
            'learning_rate': 0.05,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'min_child_weight': 3,
            'gamma': 0.1,
            'reg_alpha': 0.1,
            'reg_lambda': 1.0,
            'objective': 'binary:logistic',
            'eval_metric': ['logloss', 'aucpr'],
            'early_stopping_rounds': 20,
            'random_state': seed,
            'verbosity': 0,
        }
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    if n_pos == 0:
        raise ValueError('train_xgboost called with zero positive examples in y_train — '
                          'this is exactly the bug select_calibration_cutoff() and '
                          'walk_forward_folds() guard against; check the caller.')
    params['scale_pos_weight'] = n_neg / max(n_pos, 1)

    model = xgb.XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    return model


def get_best_iteration(model: xgb.XGBClassifier) -> int:
    """Return the best iteration, falling back to n_estimators-1 if not set."""
    if hasattr(model, 'best_iteration') and model.best_iteration is not None:
        return model.best_iteration
    return model.n_estimators - 1


def predict_best(model: xgb.XGBClassifier, X: np.ndarray) -> np.ndarray:
    """Predict probabilities using the best iteration."""
    best = get_best_iteration(model)
    return model.predict_proba(X, iteration_range=(0, best + 1))[:, 1]


def raw_margin_best(model: xgb.XGBClassifier, X: np.ndarray) -> np.ndarray:
    """Return raw margin (log-odds) from the best iteration."""
    best = get_best_iteration(model)
    booster = model.get_booster()
    dmatrix = xgb.DMatrix(X)
    return booster.predict(dmatrix, iteration_range=(0, best + 1), output_margin=True)


def fit_platt_calibrator(
    model: xgb.XGBClassifier,
    X_cal: np.ndarray,
    y_cal: np.ndarray
) -> LogisticRegression:
    """Fit Platt scaling (logistic regression on raw margins)."""
    margins = raw_margin_best(model, X_cal).reshape(-1, 1)
    platt = LogisticRegression()
    platt.fit(margins, y_cal)
    return platt


def predict_calibrated(
    model: xgb.XGBClassifier,
    platt: LogisticRegression,
    X: np.ndarray
) -> np.ndarray:
    """Calibrated probability using Platt scaling."""
    margins = raw_margin_best(model, X).reshape(-1, 1)
    return platt.predict_proba(margins)[:, 1]

In [10]:
# ----------------------------------------------------------------------------
# 10. EVALUATION AND THRESHOLD OPTIMIZATION
# ----------------------------------------------------------------------------
def evaluate_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label: str = ''
) -> Dict[str, Any]:
    """Compute a comprehensive set of evaluation metrics."""
    pred_50 = (y_pred >= 0.5).astype(int)
    result = {
        'set': label,
        'n': len(y_true),
        'n_pos': int(y_true.sum()),
        'auc_pr': round(average_precision_score(y_true, y_pred), 3),
        'auc_roc': round(roc_auc_score(y_true, y_pred), 3),
        'brier': round(brier_score_loss(y_true, y_pred), 4),
        'f1_at_50': round(f1_score(y_true, pred_50, zero_division=0), 3),
        'recall_at_50': round(recall_score(y_true, pred_50, zero_division=0), 3),
        'precision_at_50': round(precision_score(y_true, pred_50, zero_division=0), 3),
        'prob_min': round(float(y_pred.min()), 4),
        'prob_max': round(float(y_pred.max()), 4),
        'prob_std': round(float(y_pred.std()), 4),
    }
    return result


def find_optimal_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    metric: str = 'f1'
) -> float:
    """Find threshold that maximizes the given metric (f1 or youden)."""
    if metric == 'f1':
        precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
        f1_scores = 2 * (precision * recall) / (precision + recall + 1e-12)
        best_idx = np.argmax(f1_scores[:-1])  # last point has no matching threshold
        return thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    else:
        # Youden's J statistic: maximize (tpr - fpr) over the ROC curve
        fpr, tpr, thresholds_roc = roc_curve(y_true, y_prob)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        return float(thresholds_roc[best_idx])


def assign_risk_bands(
    probabilities: np.ndarray,
    high_pctile: float = RISK_HIGH_PCTILE,
    med_pctile: float = RISK_MEDIUM_PCTILE
) -> Tuple[np.ndarray, float, float]:
    """Assign HIGH/MEDIUM/LOW risk levels based on percentiles."""
    high_cut = np.percentile(probabilities, high_pctile)
    med_cut = np.percentile(probabilities, med_pctile)
    bands = np.where(
        probabilities >= high_cut, 'HIGH',
        np.where(probabilities >= med_cut, 'MEDIUM', 'LOW')
    )
    return bands, high_cut, med_cut

In [17]:
# ----------------------------------------------------------------------------
# 11. MAIN EXECUTION PIPELINE (final fix)
# ----------------------------------------------------------------------------
def main():
    """Orchestrate the entire predictive maintenance pipeline."""
    logger.info('Starting Predictive Maintenance pipeline v5.1')

    conn = get_connection()
    try:
        # 1. Get WORKORDER date range
        wo_range = run_sql(conn, f"""
            SELECT MIN(REPORTDATE) AS MIN_D, MAX(REPORTDATE) AS MAX_D
            FROM {DB_SCHEMA}.WORKORDER WHERE SITEID IN ({SITES_SQL})
        """)
        earliest = pd.to_datetime(wo_range['min_d'].iloc[0])
        latest = pd.to_datetime(wo_range['max_d'].iloc[0])
        logger.info(f'WORKORDER history: {earliest.date()} -> {latest.date()} '
                    f'({(latest-earliest).days} days)')

        # 2. Generate snapshot dates and build panel
        dates = generate_snapshot_dates(earliest, latest)
        logger.info(f'Building {len(dates)} snapshots from {dates[0].date()} to {dates[-1].date()}')
        panel = build_panel(conn, dates)

        # 3. Validate and check leakage
        validate_panel(panel)
        leakage_guard(panel)
        logger.info('Positives by snapshot date (tail 10 shown):\n' +
                    panel.groupby('as_of_date')['label'].sum().tail(10).to_string())

        # 4. Train/calibration split with very lenient parameters
        calib_cut = select_calibration_cutoff(
            panel,
            calib_positive_frac=0.02,   # only 2% of positives for calibration
            min_train_positives=1       # accept as few as 1 positive in training
        )

        train_idx = panel.index[panel['as_of_date'] <= calib_cut]
        calib_idx = panel.index[panel['as_of_date'] > calib_cut]

        X_final = panel.loc[train_idx, FEATURE_COLS].values
        y_final = panel.loc[train_idx, 'label'].values
        X_cal = panel.loc[calib_idx, FEATURE_COLS].values
        y_cal = panel.loc[calib_idx, 'label'].values

        # 5. Train final XGBoost model
        final_model = train_xgboost(X_final, y_final, X_cal, y_cal)
        logger.info(f'Final model best_iteration={final_model.best_iteration}')

        # 6. Platt scaling (only if both classes are present in calibration)
        platt = None
        if len(np.unique(y_cal)) == 2:
            platt = fit_platt_calibrator(final_model, X_cal, y_cal)
            logger.info('Platt scaling fitted successfully.')
        else:
            logger.warning('Calibration set has only one class — Platt scaling skipped. Using raw probabilities.')

        # 7. Evaluate on calibration set (raw vs calibrated if available)
        raw_proba = predict_best(final_model, X_cal)
        if platt is not None:
            cal_proba = predict_calibrated(final_model, platt, X_cal)
            logger.info('Calibration set - Calibrated metrics: ' + str(evaluate_predictions(y_cal, cal_proba)))
        logger.info('Calibration set - Raw metrics: ' + str(evaluate_predictions(y_cal, raw_proba)))

        # 8. Save model
        model_path = OUTPUT_DIR / 'xgb_pm_model_v5_1.json'
        final_model.get_booster().save_model(str(model_path))
        final_model.get_booster().set_attr(best_iteration=str(final_model.best_iteration))
        logger.info(f'Model saved to {model_path}')

        # 9. SHAP analysis
        explainer = shap.TreeExplainer(final_model)
        X_shap = panel.sample(min(1000, len(panel)))[FEATURE_COLS].values
        shap_values = explainer.shap_values(X_shap)
        shap_df = pd.DataFrame(X_shap, columns=FEATURE_COLS)

        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, shap_df, max_display=12, show=False)
        plt.title('SHAP Feature Impact (v5.1)', fontsize=12)
        plt.tight_layout()
        shap_path = OUTPUT_DIR / 'shap_global_v5_1.png'
        plt.savefig(shap_path, dpi=150, bbox_inches='tight')
        plt.show()
        logger.info(f'SHAP plot saved to {shap_path}')

        # 10. Generate predictions for today
        today = pd.Timestamp.now().normalize()
        conn_today = get_connection()
        try:
            today_snapshot = build_snapshot(conn_today, today)
        finally:
            conn_today.close()

        X_today = today_snapshot[FEATURE_COLS].values

        # Use calibrated probabilities if available, else raw
        if platt is not None:
            today_snapshot['failure_probability'] = predict_calibrated(final_model, platt, X_today)
        else:
            today_snapshot['failure_probability'] = predict_best(final_model, X_today)

        today_snapshot['failure_probability_pct'] = (today_snapshot['failure_probability'] * 100).round(1)
        bands, high_cut, med_cut = assign_risk_bands(today_snapshot['failure_probability'].values)
        today_snapshot['risk_level'] = bands
        logger.info(f'Risk cutoffs: HIGH >= {high_cut:.3f}, MEDIUM >= {med_cut:.3f}')
        logger.info('Risk level counts:\n' + today_snapshot['risk_level'].value_counts().to_string())

        # 11. Write predictions to DB2
        ddl = f"""
CREATE TABLE {DB_SCHEMA}.AI_ASSET_PREDICTION (
    ASSETNUM           VARCHAR(25)   NOT NULL,
    SITEID             VARCHAR(8)    NOT NULL,
    PREDICTIONDATE     DATE          NOT NULL,
    FAILUREPROBABILITY DECIMAL(5,2),
    RISKLEVEL          VARCHAR(10),
    MODELVERSION       VARCHAR(20),
    CREATEDATE         TIMESTAMP     NOT NULL WITH DEFAULT CURRENT TIMESTAMP,
    PRIMARY KEY (ASSETNUM, SITEID, PREDICTIONDATE)
)
"""
        insert_sql = f"""
INSERT INTO {DB_SCHEMA}.AI_ASSET_PREDICTION
    (ASSETNUM, SITEID, PREDICTIONDATE, FAILUREPROBABILITY, RISKLEVEL, MODELVERSION)
VALUES (?, ?, CURRENT DATE, ?, ?, ?)
"""
        conn_write = get_connection()
        try:
            cur = conn_write.cursor()
            try:
                cur.execute(ddl)
                conn_write.commit()
                logger.info('Table created')
            except Exception:
                logger.info('Table already exists')

            cur.execute(f'DELETE FROM {DB_SCHEMA}.AI_ASSET_PREDICTION WHERE PREDICTIONDATE = CURRENT DATE')
            inserted = 0
            for _, row in today_snapshot.iterrows():
                try:
                    cur.execute(insert_sql, (
                        row['assetnum'], row['siteid'],
                        float(row['failure_probability_pct']),
                        row['risk_level'], MODEL_VERSION
                    ))
                    inserted += 1
                except Exception as e:
                    logger.error(f'Failed to insert {row["assetnum"]}: {e}')
            conn_write.commit()
            cur.close()
            logger.info(f'Inserted {inserted} predictions into {DB_SCHEMA}.AI_ASSET_PREDICTION')
        finally:
            conn_write.close()

        # 12. Save CSV backup
        csv_path = OUTPUT_DIR / 'predictions_v5_1.csv'
        today_snapshot.to_csv(csv_path, index=False)
        logger.info(f'Predictions saved to {csv_path}')

    finally:
        conn.close()

    logger.info('Pipeline completed successfully.')

In [ ]:
if __name__ == '__main__':
    main()

2026-07-12 16:20:52  INFO  Starting Predictive Maintenance pipeline v5.1
2026-07-12 16:20:53  INFO  WORKORDER history: 2023-02-08 -> 2026-06-22 (1230 days)
2026-07-12 16:20:53  INFO  Building 396 snapshots from 2023-02-22 to 2026-05-22
2026-07-12 16:20:54  INFO    2023-02-22: 1,833 assets, 0 positive
2026-07-12 16:20:55  INFO    2023-02-25: 1,833 assets, 0 positive
2026-07-12 16:20:56  INFO    2023-02-28: 1,833 assets, 0 positive
2026-07-12 16:20:57  INFO    2023-03-03: 1,833 assets, 0 positive
2026-07-12 16:20:58  INFO    2023-03-06: 1,833 assets, 0 positive
2026-07-12 16:20:59  INFO    2023-03-09: 1,833 assets, 0 positive
2026-07-12 16:20:59  INFO    2023-03-12: 1,833 assets, 0 positive
2026-07-12 16:21:00  INFO    2023-03-15: 1,833 assets, 0 positive
2026-07-12 16:21:01  INFO    2023-03-18: 1,833 assets, 0 positive
2026-07-12 16:21:02  INFO    2023-03-21: 1,833 assets, 0 positive
2026-07-12 16:21:03  INFO    2023-03-24: 1,833 assets, 0 positive
2026-07-12 16:21:04  INFO    2023-03-2